# Test existing plugins

## 1. Test Google Scholar / Serpapi plugin with one DOI

In [2]:
import logging
import os
from doi_downloader import loader as ld, pdf_download as pdf_dl
from habanero import Crossref

logging.basicConfig(level=logging.INFO)  

In [3]:
doi = "10.1038/s41586-025-10047-5"
correct_pdf_url = "https://www.nature.com/articles/s41586-025-10047-5.pdf"
outfile_name = "pdf_file.pdf"

In [18]:
def get_metadata(doi):
    """Get metadata from crossref"""
    crossref_api = Crossref(mailto="e.f.tjongkimsang@tilburguniversity.edu")          # polite pool, set once
    crossref_data = crossref_api.works(ids=doi)
    author = crossref_data["message"]["author"][0]["family"]
    title = crossref_data["message"]["title"][0]
    year = crossref_data["message"]["created"]["date-parts"][0][0]
    return author, title, year

In [19]:
author, title, year = get_metadata(doi)

INFO:httpx2:HTTP Request: GET https://api.crossref.org/works/10.1038/s41586-025-10047-5 "HTTP/1.1 200 OK"


Retrieve description file for DOI from https://doi.org and check if it is correct

In [3]:
pdf_urls = ld.plugins["GoogleScholarSerpAPIPlugin"].get_pdf_urls(doi, read_from_cache=False, save_to_cache=True)
assert pdf_urls == [correct_pdf_url]

INFO:plugins.googlescholar:[serpapi] ✅ Found DOI 10.1038/s41586-025-10047-5 in metadata


Download PDF file from the url, check if the download was successful and verify that the PDF mentions the DOI

In [4]:
try: os.remove(outfile_name)
except FileNotFoundError: pass
path_name, pdf_has_doi = pdf_dl.download_pdf(pdf_urls[0], outfile_name, ".", plugin_name="doiorg", doi=doi)
assert os.path.isfile(outfile_name), "Missing downloaded pdf file"
assert os.path.getsize(outfile_name) > 0, "Downloaded file is empty"

[doiorg] ✅ Found DOI in PDF on page 1


## 2. Test all plugins with multiple DOIs

Some of these plugins need an access key and will fail without such a key

### 2.1 Support functions

In [ ]:
from collections import Counter
from doi_downloader import loader as ld, doi_downloader as ddl, pdf_download as pdf_dl
import json
import os
import polars as pl
import regex
from termcolor import colored
import time
import urllib

In [ ]:
plugins = ld.plugins
PLUGINS = {"CoreacukPlugin": plugins['CoreacukPlugin'],
           "CrossrefPlugin": plugins['CrossrefPlugin'],
           "DoiorgPlugin": plugins['DoiorgPlugin'],
           "GoogleScholarSerpAPIPlugin": plugins['GoogleScholarSerpAPIPlugin'],
           "UnpaywallPlugin": plugins['UnpaywallPlugin']} # requires a commercial api key

CHAR_SUCCESS = "✅"
CHAR_FAILURE = "❌"

In [ ]:
def make_safe_filename(doi):
    """Replace slashes and points in DOI by underscores, Add .pdf"""
    return doi.replace("/", "_").replace(".", "_") + ".pdf"

In [ ]:
async def get_pdf_urls_from_all_plugins(doi, debug=False):
    """Retrieve pdf_urls with all plugins"""
    urls = {}
    for plugin_name in PLUGINS.keys():
        if plugin_name == "GoogleScholarSerpAPIPlugin":
            try:
                plugin_data = await PLUGINS[plugin_name].get_pdf_urls(doi, read_from_cache=True, save_to_cache=False) \
                              if plugin_name == "GoogleScholarSerpAPIPlugin" \
                              else PLUGINS[plugin_name].get_pdf_url(doi, read_from_cache=True, save_to_cache=False)
                urls[plugin_name] = plugin_data if type(plugin_data) == list else [plugin_data] if type(plugin_data) == str else [] 
            except Exception as e:
                print(f"   get_pdf_urls_from_all_plugins: error: {e}")
    return urls

In [ ]:
def summarize_plugin_results(pdf_urls, index, result_type="url"):
    """Summarize plugin results with single icon: CHAR_SUCCESS or CHAR_FAILURE"""
    prefix =  f"   " if result_type == "url" else "   "
    print(f"{prefix} {result_type} retrieval success per plugin:", end=" ")
    for plugin_name in sorted(pdf_urls):
        print((CHAR_SUCCESS if pdf_urls[plugin_name] else CHAR_FAILURE), end=" ")
    print()

In [ ]:
plugin_names = {"GoogleScholarSerpAPIPlugin": "serpapi"}

def download_pdf(doi, pdf_target_dir_name, pdf_target_file_name, pdf_urls):
    """Try to download pdf with each plugin, Register result"""
    target_locations = {}
    pdf_has_doi = {}
    for plugin_name in pdf_urls:
        if pdf_urls[plugin_name]:
            for pdf_url in pdf_urls[plugin_name]:
                (target_locations[plugin_name], 
                 pdf_has_doi[plugin_name]) = pdf_dl.download_pdf(pdf_url, 
                                                                 pdf_target_file_name, 
                                                                 pdf_target_dir_name,
                                                                 plugin_name=plugin_names[plugin_name],
                                                                 doi=doi)
    #for plugin_name in PLUGINS:
    #    if plugin_name not in target_locations or not target_locations[plugin_name]:
    #        target_locations[plugin_name] = None
    #        pdf_has_doi[plugin_name] = False
    return target_locations, pdf_has_doi

In [ ]:
async def check_plugin_access(doi_df):
    """Check plugin access to a list of DOIs: url retrieval and pdf downloads"""
    target_locations = []
    pdf_has_doi = []
    pdf_urls = []
    pdf_target_dir_name = "."
    for index, row in enumerate(doi_df.iter_rows()):
        doi = row[0]
        print(f"{index + 1}. {doi}")
        pdf_target_file_name = make_safe_filename(doi)
        pdf_urls.append(await get_pdf_urls_from_all_plugins(doi, debug=False))
        summarize_plugin_results({key[0]: pdf_urls[-1][key[0]] 
                                  for key in sorted(pdf_urls[-1].items(), 
                                                    key=lambda item: item[0])}, 
                                 index + 1)
        target_locations_dict, pdf_has_doi_dict = (download_pdf(doi,
                                                                pdf_target_dir_name, 
                                                                pdf_target_file_name, 
                                                                pdf_urls[-1]))
        target_locations.append(target_locations_dict)
        pdf_has_doi.append(pdf_has_doi_dict)
        summarize_plugin_results({key[0]: target_locations[-1][key[0]] 
                                          for key in sorted(target_locations[-1].items(), 
                                                     key=lambda item: item[0])}, 
                                          index + 1, result_type="pdf")
        if "MAX_PROCESSED" in globals() and index >= MAX_PROCESSED:
             break
    return pdf_urls, target_locations, pdf_has_doi

In [ ]:
def count_successes(pdf_urls):
    """Count the number of successes per plugin and their combination"""
    counts = Counter()
    for row in pdf_urls:
        for plugin_name, value in row.items():
            counts[plugin_name] += 1 if value else 0
        counts["any"] += 1 if any(row.values()) else 0 
    counts_df = pl.DataFrame(counts)
    return counts_df.select(sorted(counts_df.columns))

### 2.2 browser plugin test set (14)

In [ ]:
doi_plugin_df = pl.DataFrame([{"doi": "10.1613/jair.49"},
                              {"doi": "10.1038/s41586-025-10047-5"},
                              {"doi": "10.3390/electronics15040795"},
                              {"doi": "10.3389/fpsyt.2025.1739639"},
                              {"doi": "10.4236/jhrss.2026.141006"},
                              {"doi": "10.3897/aiep.51.63489"},
                              {"doi": "10.1016/j.nlp.2026.100202"},
                              {"doi": "10.1177/0022002714560349"},
                              {"doi": "10.1007/s10198-013-0496-x"},
                              {"doi": "10.1111/j.1465-7295.2010.00309.x"},
                              {"doi": "10.1016/j.econlet.2009.08.024"},
                              {"doi": "10.1093/ei/cb1001"},
                              {"doi": "10.2174/2213476X07666200423081738"},
                              {"doi": "10.1504/EJIM.2025.150039"}])

In [ ]:
MAX_PROCESSED = 20

In [ ]:
pdf_urls, target_locations, pdf_has_doi = await check_plugin_access(doi_plugin_df)

In [ ]:
count_successes(pdf_urls)

In [ ]:
count_successes(target_locations)

In [ ]:
count_successes(pdf_has_doi)

Google Scholar PDF downloads without any support by DOI mentions: 3/6: 
* (1) 10.1613/jair.49
* (11) 10.1016/j.econlet.2009.08.024
* (12) 10.1093/ei/cb1001

### 2.3 doi_coda.csv (top 20)

The test DOIs were taken from the file `doi_coda.csv` (first twenty)

In [ ]:
doi_coda_df = pl.DataFrame([{"doi": '10.1177/0022002714560349'},
                            {"doi": '10.1007/s10198-013-0496-x'},
                            {"doi": '10.1111/j.1465-7295.2010.00309.x'},
                            {"doi": '10.1016/j.econlet.2009.08.024'},
                            {"doi": '10.1038/s41467-017-00731-0'},
                            {"doi": '10.1093/ei/cbl001'},
                            {"doi": '10.1016/j.jesp.2004.09.004'},
                            {"doi": '10.1016/j.jpubeco.2014.01.005'},
                            {"doi": '10.1111/puar.12424'},
                            {"doi": '10.1016/j.geb.2006.09.005'},
                            {"doi": '10.1016/j.joep.2007.01.007'},
                            {"doi": '10.1108/03068290810854565'},
                            {"doi": '10.1111/j.1468-0335.2008.00689.x'},
                            {"doi": '10.1177/1069397108322455'},
                            {"doi": '10.1016/j.socec.2010.12.013'},
                            {"doi": '10.1017/s0022381609090355'},
                            {"doi": '10.1016/j.jpubeco.2008.06.007'},
                            {"doi": '10.1093/restud/rdt017'},
                            {"doi": '10.1016/j.socec.2011.08.028'},
                            {"doi": '10.1093/jae/ejr034'}])          

In [ ]:
MAX_PROCESSED = 20

In [ ]:
pdf_urls, target_locations, pdf_has_doi = await check_plugin_access(doi_coda_df[:MAX_PROCESSED])

In [ ]:
count_successes(pdf_urls)

In [ ]:
count_successes(target_locations)

In [ ]:
count_successes(pdf_has_doi)

Google Scholar PDF downloads without any support by DOI mentions: 2/4: 10.1016/j.econlet.2009.08.024 and 10.1093/ei/cbl001

## 3. In-depth testing of cause of pdf url retrieval failure with doi.org plugin

In [ ]:
from doi_downloader.plugins.doiorg import DoiorgPlugin
import requests
from termcolor import colored
from time import sleep

In [ ]:
URL_ACCESS_WAIT_TIME = 7

In [ ]:
plugin = DoiorgPlugin()
cache = {}

def robot_access_allowed(url):
    if url not in cache:
        sleep(URL_ACCESS_WAIT_TIME)
        cache[url] = access_allowed = plugin.robot_access_allowed(url)
    return cache[url]

In [ ]:
def color_code_status(status_code):
    if status_code == 200:
        return colored(status_code, "green")
    elif status_code >= 400:
        return colored(status_code, "red")
    else:
        return status_code

In [ ]:
headers = { "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                           "AppleWebKit/537.36 (KHTML, like Gecko) "
                           "Chrome/120.0.0.0 Safari/537.36"),}

doi_df = doi_coda_df.clone()
status_code_counts = {}
for index, doi in enumerate(doi_df.get_columns()[0]):
    print(f"{index + 1:2d}. {doi}")
    sleep(URL_ACCESS_WAIT_TIME)
    response = requests.get("https://doi.org/" + doi, headers=headers, timeout=10)
    for r in response.history + [response]:
        print("   ", color_code_status(r.status_code), robot_access_allowed(r.url), r.url)
    if r.status_code not in status_code_counts:
        status_code_counts[r.status_code] = 0
    status_code_counts[r.status_code] += 1
status_code_counts

**Result coda DOIs**: ten of 20 DOIs result in a webpage retrieval with status code 200 (Success) while the other ten (1, 3, 6, 9, 12, 13, 14, 16, 18 and 20) produce a status code with value 403 (Access forbidden)<br>
**Result browser plugin DOIs**: eight of 14 DOIs result in a webpage retrieval with status code 200 (Success) while the other six (3, 5, 8, 10, 12 and 13) produce a status code with value 403 (Access forbidden)<br>
Results may vary because of unpredictable website behaviour

## 4. Playwright tests

In [ ]:
import asyncio
from playwright.async_api import async_playwright

In [ ]:
async def get_page(url):
    async with async_playwright() as p:
        browser = await p.firefox.launch()
        page = await browser.new_page()
        history = []
        page.on("response", lambda r: history.append((r.status, r.url)))
        response = await page.goto(url)
        content = await page.content()
        await browser.close()
        return response, content, history

In [ ]:
doi_df = doi_coda_df.clone()
status_counts = {}
for index, doi in enumerate(list(doi_df.get_columns()[0])):
    print(f"{index + 1:2d}. {doi}")
    sleep(URL_ACCESS_WAIT_TIME)
    response, content, history = await get_page("https://doi.org/" + doi)
    print("   ", color_code_status(response.status), robot_access_allowed(response.url), response.url)
    if response.status not in status_counts:
        status_counts[response.status] = 0
    status_counts[response.status] += 1
status_counts

**Result coda DOIs** thirteen of 20 DOIs result in a webpage retrieval with status code 200 (Success) while the other seven (1, 3, 6, 9, 13, 14 and 16) produced a status code with value 403 (Access forbidden)<br>
**Result browser plugin DOIs** nine of 14 DOIs result in a webpage retrieval with status code 200 (Success) while four (8, 10, 12 and 14) produced a status code with value 403 (Access forbidden). One DOI (5) resulted in a time-out<br>
Results may vary because of unpredictable website behaviour

## 5. Summary

Success in PDF URL retrieval (columns 2-9) and PDF retrieval (columns 10-14) for the coda data set per DOI:

| id | coreacuk | crossref | doiorg | google | unpaywall | status code | playwright | difference | coreacuk | crossref | doiorg | google | unpaywall |
|:--:|:--------:|:--------:|:------:|:------:|:---------:|:-----------:|:----------:|:----------:|:--------:|:--------:|:------:|:------:|:---------:|
|  1 | ❌ | ✅ | ❌ | ✅ | ❌ | 403 | 403 | |
|  2 | ❌ | ✅ | ❌ | ✅ | ❌ | 200 | 200 | |
|  3 | ❌ | ❌ | ❌ | ✅ | ❌ | 403 | 403 | | | | | ✅ | |
|  4 | ❌ | ❌ | ❌ | ✅ | ❌ | 200 | 200 | | | | | ✅ | |
|  5 | ❌ | ✅ | ✅ | ✅ | ✅ | 200 | 200 | | | ✅ | ✅ | ✅ | ✅ |
|  6 | ❌ | ❌ | ❌ | ✅ | ✅ | 403 | 403 | | | | | ✅ | ✅ |
|  7 | ❌ | ❌ | ❌ | ✅ | ❌ | 200 | 200 | |
|  8 | ❌ | ❌ | ❌ | ✅ | ❌ | 200 | 200 | |
|  9 | ❌ | ✅ | ❌ | ✅ | ✅ | 403 | 403 | |
| 10 | ❌ | ❌ | ❌ | ✅ | ✅ | 200 | 200 | | | | | | ✅ |
| 11 | ❌ | ❌ | ❌ | ✅ | ❌ | 200 | 200 | |
| 12 | ❌ | ❌ | ❌ | ✅ | ❌ | 403 | 200 |!|
| 13 | ❌ | ❌ | ❌ | ✅ | ❌ | 403 | 403 | |
| 14 | ❌ | ✅ | ❌ | ✅ | ❌ | 403 | 403 | |
| 15 | ❌ | ❌ | ❌ | ✅ | ❌ | 200 | 200 | |
| 16 | ❌ | ❌ | ❌ | ✅ | ❌ | 403 | 403 | |
| 17 | ❌ | ❌ | ❌ | ✅ | ❌ | 200 | 200 | |
| 18 | ❌ | ❌ | ❌ | ✅ | ❌ | 403 | 200 |!|
| 19 | ❌ | ❌ | ❌ | ✅ | ❌ | 200 | 200 | |
| 20 | ❌ | ❌ | ❌ | ✅ | ❌ | 403 | 200 |!| | | | ✅ | |
|    |  0 |  5 |  1 | 20 |  4 |  10  | 13  |3|0|1|1|5|3| 

Success in PDF URL retrieval (columns 2-9) and PDF retrieval (columns 10-14) for the browser plugin data set per DOI:

| id | coreacuk | crossref | doiorg | google | unpaywall | status code | playwright | difference | coreacuk | crossref | doiorg | google | unpaywall |
|:--:|:--------:|:--------:|:------:|:------:|:---------:|:-----------:|:----------:|:----------:|:--------:|:--------:|:------:|:------:|:---------:|
|  1 | ❌ | ✅ | ✅ | ✅ | ✅ | 200 | 200 | | | ✅ | ✅ | ✅ | ✅ |
|  2 | ❌ | ✅ | ✅ | ✅ | ❌ | 200 | 200 | | | ✅ | ✅ | ✅ | |
|  3 | ❌ | ❌ | ❌ | ✅ | ✅ | 403 | 200 |!|
|  4 | ❌ | ❌ | ✅ | ✅ | ✅ | 200 | 200 | | | | ✅ | | ✅|
|  5 | ❌ | ✅ | ❌ | ✅ | ✅ | 403 | 200 |!| | | | | |
|  6 | ❌ | ✅ | ✅ | ✅ | ❌ | 200 | 200 | | | ✅ | ✅ | ✅ | |
|  7 | ❌ | ❌ | ❌ | ✅ | ❌ | 200 | 200 | |
|  8 | ❌ | ✅ | ❌ | ✅ | ❌ | 403 | 403 | |
|  9 | ❌ | ✅ | ❌ | ✅ | ❌ | 200 | 200 | |
| 10 | ❌ | ❌ | ❌ | ✅ | ❌ | 403 | 403 | | | | | ✅ | |
| 11 | ❌ | ❌ | ❌ | ✅ | ❌ | 200 | 200 | | | | | ✅ | |
| 12 | ❌ | ❌ | ❌ | ✅ | ✅ | 403 | 403 |!| | | | ✅ | ✅ |
| 13 | ❌ | ✅ | ❌ | ✅ | ❌ | 403 | 200 | |
| 14 | ❌ | ❌ | ❌ | ✅ | ❌ | 200 | 403 |?|
|    |  0 |  7 |  4 | 14 |  5 |   6  |  4  |4|0|3|4|6|4|
